In [9]:
import torch
from torch import nn
from torch.nn import functional as F

In [17]:
# self define a convolution:
#padding=0,stride=1
'''二维互相关运算：correlation 2d'''  
def corr2d(X,K):
    nk,nd=K.shape  #have  kernel(fliter)'s shape
    mk,md=X.shape  # have the input's shape
    Y=torch.zeros((mk-nk+1),(md-nd+1))
    #start calculate
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i,j]=(X[i:i+nk,j:j+nd]*K).sum()  #*恰好是对应元素相乘，符合卷积要求
                    #只使用一个方括号而非两个方括号X[][]
                    #两个方括号只相当于对X两次行提取
                    #torch中矩阵维度提取，每次行列都在同一个中括号内用,隔开
    return Y

In [23]:
x=torch.tensor([[1.,2.,3.],[4.,5.,6.],[7.,8.,9.]])
k=torch.tensor([[2.,4.],[1.,8.]])
y=corr2d(x,k)
x,k,y

(tensor([[1., 2., 3.],
         [4., 5., 6.],
         [7., 8., 9.]]),
 tensor([[2., 4.],
         [1., 8.]]),
 tensor([[ 54.,  69.],
         [ 99., 114.]]))

In [30]:
#2dim convolytion layer
class Conv2D(nn.Module):
    def __init__(self,kernel_size):
        super().__init__()
        self.weight=nn.Parameter(torch.rand(kernel_size))  #matrix formally
        self.bias=nn.Parameter(torch.zeros(1))
    def forward(self,x):
        return corr2d(x,self.weight)+self.bias      #call the 2D cross-correlation function
                                                    #self.weight act as K


In [36]:
net=Conv2D([2,2])  # the kernel size must be a 2D tensor [a,b]  (a tuple)
net(x)   #equal to net.forward(x),the lib will call __call(parameters)__

tensor([[3.8825, 5.2907],
        [8.1073, 9.5156]], grad_fn=<AddBackward0>)

if writes:
```python
        net=Conv2D(2)
        net(x)

```
the dimision of kernel_size is wrong
the error infomation is as follows:
## always read from the bottom
Traceback will display the full calling chain (out to inner)
so the error is from the bottom(inner)  
&nbsp;
&nbsp;
&nbsp;
&nbsp;
&nbsp;
&nbsp;
&nbsp;
```
ValueError                                Traceback (most recent call last)
Cell In[34], line 2
      1 net=Conv2D(2)  # the kernel size must be a 2D tensor [a,b]
----> 2 net(x)   #equal to net.forward(x),the lib will call __call(parameters)__

File C:\DL_code\.venv\Lib\site-packages\torch\nn\modules\module.py:1778, in Module._wrapped_call_impl(self, *args, **kwargs)
   1776     return self._compiled_call_impl(*args, **kwargs)  # type: ignore[misc]
   1777 else:
-> 1778     return self._call_impl(*args, **kwargs)

File C:\DL_code\.venv\Lib\site-packages\torch\nn\modules\module.py:1789, in Module._call_impl(self, *args, **kwargs)
   1784 # If we don't have any hooks, we want to skip the rest of the logic in
   1785 # this function, and just call forward.
   1786 if not (self._backward_hooks or self._backward_pre_hooks or self._forward_hooks or self._forward_pre_hooks
   1787         or _global_backward_pre_hooks or _global_backward_hooks
   1788         or _global_forward_hooks or _global_forward_pre_hooks):
-> 1789     return forward_call(*args, **kwargs)
   1791 result = None
   1792 called_always_called_hooks = set()

Cell In[30], line 8, in Conv2D.forward(self, x)
      7     def forward(self,x):
----> 8         return corr2d(x,self.weight)+self.bias      #call the 2D cross-correlation function
      9                                                     #self.weight act as K

Cell In[17], line 5, in corr2d(X, K)
      4 def corr2d(X,K):
----> 5     nk,nd=K.shape  #have  kernel(fliter)'s shape
      6     mk,md=X.shape  # have the input's shape
      7     Y=torch.zeros((mk-nk+1),(md-nd+1))
      8     #start calculate

ValueError: not enough values to unpack (expected 2, got 1)
```

the bottom error says:"not enough values to unpack"
and the last error_info points to  

```python
nk,nd=K.shape  #have  kernel(fliter)'s shape
```
it seems that the shape of kernel K,can’t be unpack successfully  
so check your set of K,as to kernel_size  
that's to say  
```python
        net=Conv2D(2)
```
is the error